Plot skill maps


In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

CACHE_DIR = Path("../../results/model/evaluation/gimms/cache")
FP_EVAL = CACHE_DIR / "eval_results_boundexp_siam_kge08_cv5.csv"
FP_PILOT = CACHE_DIR / "pilot_results_boundexp_siam_kge08_coarse2093_cv5.csv"

print("CACHE_DIR:", CACHE_DIR.resolve())
print("exists eval:", FP_EVAL.exists(), "| pilot:", FP_PILOT.exists())


In [ ]:
fp_load = FP_EVAL if FP_EVAL.exists() else FP_PILOT
if not fp_load.exists():
    raise FileNotFoundError(
        f"Missing evaluation CSVs under {CACHE_DIR}. "
        "Run evaluate_gimms_test2.ipynb Fit/CV first."
    )

evaluation_results = pd.read_csv(fp_load)
print("Loaded", fp_load)
print("n rows:", len(evaluation_results), "| n grids:", evaluation_results["pixel"].nunique())
print("models:", sorted(evaluation_results["model"].unique().tolist()))
print("columns:", list(evaluation_results.columns))
evaluation_results.head()


In [ ]:
def best_metric(data, models=None):
        df = data.copy()
    if models is None:
        models = ["CDD", "CDDP", "SIAM", "SIAMP"]
        models = [m for m in models if m in set(df["model"])]

    wide = (
        df.pivot_table(
            index=["pixel", "latitude", "longitude"],
            columns="model",
            values=["rmse", "r", "kge"],
            aggfunc="first",
        )
        .sort_index(axis=1)
    )
    wide.columns = [f"{model}_{metric}" for metric, model in wide.columns]
    out = wide.reset_index()

    rmse_cols = [f"{m}_rmse" for m in models if f"{m}_rmse" in out.columns]
    r_cols = [f"{m}_r" for m in models if f"{m}_r" in out.columns]
    kge_cols = [f"{m}_kge" for m in models if f"{m}_kge" in out.columns]

    out["lowest_rmse"] = out[rmse_cols].idxmin(axis=1).str.replace("_rmse", "", regex=False)
    out["highest_r"] = out[r_cols].idxmax(axis=1).str.replace("_r", "", regex=False)
    out["highest_kge"] = out[kge_cols].idxmax(axis=1).str.replace("_kge", "", regex=False)
    return out


In [ ]:
import matplotlib.pyplot as plt
import rasterio as rs
from rasterio.features import rasterize
from rasterio.transform import from_origin
import geopandas as gpd
import numpy as np
import cartopy.crs as ccrs
import cartopy.io.shapereader as shpreader
from shapely.geometry import box
import matplotlib.path as mpath
from matplotlib.colors import ListedColormap

def rasterize_best_model(gdf, transform, width, height, model_classes, metric):
    shapes = (
        (geom, model_classes[model])
        for geom, model in zip(gdf.geometry, gdf[metric])
    )
    model_raster = rasterize(
        shapes=shapes,
        out_shape=(height, width),
        transform=transform,
        fill=np.nan,
        dtype="float32",
    )
    return model_raster

def show_best_model_map(
    df, metric,
    tiff_path="../../data/satellite_data/images/base-image/test.tif",
    mode="scale_xy",            # "absolute", "scale", or "scale_xy"
    res_x=0.25,                 # used for "absolute": pixel width in CRS units
    res_y=0.25,                 # used for "absolute": pixel height in CRS units
    scale_factor=2.0,           # used for "scale": same factor for both axes
    scale_factor_x=12.5,        # used for "scale_xy": longitude (x) factor
    scale_factor_y=9.4,         # used for "scale_xy": latitude (y) factor
    continents=None             # e.g., ["North America", "Europe"]; None = all land
):
    with rs.open(tiff_path) as src:
        crs = src.crs
        left, bottom, right, top = src.bounds
        transform0 = src.transform

    orig_res_x = transform0.a
    orig_res_y = abs(transform0.e)

    if mode == "absolute":
        px_x = res_x
        px_y = res_y
    elif mode == "scale":
        px_x = orig_res_x * scale_factor
        px_y = orig_res_y * scale_factor
    elif mode == "scale_xy":
        px_x = orig_res_x * scale_factor_x
        px_y = orig_res_y * scale_factor_y
    else:
        raise ValueError("mode must be 'absolute', 'scale', or 'scale_xy'")

    width = int(np.ceil((right - left) / px_x))
    height = int(np.ceil((top - bottom) / px_y))
    transform = from_origin(left, top, px_x, px_y)

    print(f"[DEBUG] Original pixel size: x={orig_res_x}, y={orig_res_y} (CRS units)")
    print(f"[DEBUG] New pixel size: x={px_x}, y={px_y} (CRS units)")
    print(f"[DEBUG] Output grid: width={width}, height={height}")

    model_colors = {
        'CDD': '#A3D4E0',
        'CDDP': '#5BAED8',
        'SIAM': '#3B83C4',
        'SIAMP': '#D1837D',
    }
    model_classes = {name: i for i, name in enumerate(model_colors.keys())}

    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df.longitude, df.latitude),
        crs=crs
    )

    raster_bounds = box(left, bottom, right, top)
    gdf = gdf[gdf.geometry.within(raster_bounds)]
    print(f"[DEBUG] Points inside raster bounds: {len(gdf)}")

    model_raster = rasterize_best_model(gdf, transform, width, height, model_classes, metric)

    ne_path = shpreader.natural_earth(
        resolution="110m",
        category="cultural",
        name="admin_0_countries"
    )
    countries = gpd.read_file(ne_path)  # CRS: EPSG:4326

    if continents is not None:
        countries = countries[countries["CONTINENT"].isin(continents)]

    land_poly = countries.union_all()
    land_gdf = gpd.GeoDataFrame(geometry=[land_poly], crs="EPSG:4326")

    land_gdf = land_gdf.to_crs(crs)

    land_gdf = gpd.clip(land_gdf, gpd.GeoDataFrame(geometry=[raster_bounds], crs=crs))

    land_mask = rasterize(
        [(geom, 1) for geom in land_gdf.geometry],
        out_shape=(height, width),
        transform=transform,
        fill=0,
        dtype="uint8",
    )

    model_raster = np.where(land_mask == 1, model_raster, np.nan)

    fig = plt.figure(figsize=[6, 6])
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.NorthPolarStereo())
    ax.set_extent([-180, 180, 30, 90], ccrs.PlateCarree())
    ax.coastlines()

    theta = np.linspace(0, 2 * np.pi, 100)
    verts = np.vstack([np.sin(theta), np.cos(theta)]).T
    circle = mpath.Path(verts * 0.5 + [0.5, 0.5])
    ax.set_boundary(circle, transform=ax.transAxes)

    cmap = ListedColormap([model_colors[m] for m in model_classes.keys()])
    cmap.set_bad((0, 0, 0, 0))  # Make NaN transparent

    ax.imshow(
        np.ma.masked_invalid(model_raster),
        cmap=cmap,
        extent=[left, right, bottom, top],
        transform=ccrs.PlateCarree(),
        origin="upper",
        interpolation="nearest",
    )

    import matplotlib.ticker as mticker
    from cartopy.mpl.ticker import LatitudeFormatter, LongitudeFormatter
    gl = ax.gridlines(linewidth=0.5, color='gray', alpha=0.7, linestyle='--')
    gl.xlocator = mticker.FixedLocator(np.arange(-180, 181, 60))
    gl.ylocator = mticker.FixedLocator([30, 50, 70])
    lat_formatter = LatitudeFormatter()
    lon_formatter = LongitudeFormatter()
    for lat in [30, 50, 70]:
        ax.text(180, lat + 5, lat_formatter(lat),
                transform=ccrs.PlateCarree(), ha='left', va='center',
                fontsize=12, color='black')
    for lon in np.arange(-180, 181, 60):
        if lon == -180:  # same meridian as 180°
            continue

        label_lon, label_lat = lon, 25  # adjust position

        if lon in [-120, 120]:
            label_lat = 23
        if lon == -60:
            label_lat = 25
        if lon == 0:
            label_lon = 2
            label_lat = 29
        if lon == 180:
            label_lon = 178

        ax.text(label_lon, label_lat, lon_formatter(lon),
                transform=ccrs.PlateCarree(),
                ha='center', va='top', fontsize=12, color='black')
        
    valid_pixels = np.isfinite(model_raster).sum()
    percentages = []
    
    for model, idx in model_classes.items():
        count = np.sum(model_raster == idx)
        pct = (count / valid_pixels) * 100 if valid_pixels > 0 else 0
        percentages.append(pct)
    
    inset_ax = fig.add_axes([0.12, 0.10, 0.35, 0.25])  # [left, bottom, width, height]
    
    inset_ax.bar(
        model_classes.keys(),
        percentages,
        color=[model_colors[m] for m in model_classes.keys()],
        edgecolor="none"
    )
    
    inset_ax.tick_params(axis="x", labelrotation=45, labelsize=10)
    inset_ax.tick_params(axis="y", labelsize=10)
    inset_ax.set_ylabel("Percentage (%)", fontsize=10)

    inset_ax.spines['top'].set_visible(False)
    inset_ax.spines['right'].set_visible(False)

    # plt.tight_layout()
    # plt.show()
    return fig


In [ ]:
df_evaluation = best_metric(evaluation_results)

In [ ]:
fig = show_best_model_map(df_evaluation, 'lowest_rmse')
fig.savefig("../../results/si_figures/si_fig5/lowest_rmse.png", dpi=300, bbox_inches='tight')

In [ ]:
fig = show_best_model_map(df_evaluation, 'highest_r')
fig.savefig("../../results/si_figures/si_fig5/highest_r.png", dpi=300, bbox_inches='tight')


In [ ]:
fig = show_best_model_map(df_evaluation, 'highest_kge')
fig.savefig("../../results/si_figures/si_fig5/highest_kge.png", dpi=300, bbox_inches='tight')


## RMSE / KGE maps


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import rasterio as rs
from rasterio.features import rasterize
from rasterio.transform import from_origin
import geopandas as gpd
import cartopy.crs as ccrs
import cartopy.io.shapereader as shpreader
from shapely.geometry import box
import matplotlib.path as mpath
from cartopy.mpl.ticker import LatitudeFormatter, LongitudeFormatter
from matplotlib.colors import Normalize, LinearSegmentedColormap

FIG1_CMAP = LinearSegmentedColormap.from_list(
    "fig1_cmap",
    ["#0b3c68", "#165188", "#2066a8", "#4d91c4", "#8ec1da",
     "#fbebe1", "#f6d6c2", "#d47264", "#c14d48", "#ae282c"],
)

def _polar_land_setup(
    tiff_path="../../data/satellite_data/images/base-image/test.tif",
    scale_factor_x=12.5,
    scale_factor_y=9.4,
):
    with rs.open(tiff_path) as src:
        crs = src.crs
        left, bottom, right, top = src.bounds
        transform0 = src.transform
    px_x = transform0.a * scale_factor_x
    px_y = abs(transform0.e) * scale_factor_y
    width = int(np.ceil((right - left) / px_x))
    height = int(np.ceil((top - bottom) / px_y))
    transform = from_origin(left, top, px_x, px_y)

    ne_path = shpreader.natural_earth(
        resolution="110m", category="cultural", name="admin_0_countries"
    )
    countries = gpd.read_file(ne_path)
    land_poly = countries.union_all()
    land_gdf = gpd.GeoDataFrame(geometry=[land_poly], crs="EPSG:4326").to_crs(crs)
    raster_bounds = box(left, bottom, right, top)
    land_gdf = gpd.clip(land_gdf, gpd.GeoDataFrame(geometry=[raster_bounds], crs=crs))
    land_mask = rasterize(
        [(geom, 1) for geom in land_gdf.geometry],
        out_shape=(height, width),
        transform=transform,
        fill=0,
        dtype="uint8",
    )
    return dict(
        crs=crs, left=left, bottom=bottom, right=right, top=top,
        transform=transform, width=width, height=height, land_mask=land_mask,
    )

def rasterize_metric(df, value_col, setup):
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df.longitude, df.latitude),
        crs=setup["crs"],
    )
    raster_bounds = box(setup["left"], setup["bottom"], setup["right"], setup["top"])
    gdf = gdf[gdf.geometry.within(raster_bounds) & np.isfinite(gdf[value_col])]
    shapes = ((geom, float(val)) for geom, val in zip(gdf.geometry, gdf[value_col]))
    rast = rasterize(
        shapes=shapes,
        out_shape=(setup["height"], setup["width"]),
        transform=setup["transform"],
        fill=np.nan,
        dtype="float32",
    )
    return np.where(setup["land_mask"] == 1, rast, np.nan)

def _decorate_polar(ax):
    ax.set_extent([-180, 180, 30, 90], ccrs.PlateCarree())
    ax.coastlines()
    theta = np.linspace(0, 2 * np.pi, 100)
    verts = np.vstack([np.sin(theta), np.cos(theta)]).T
    ax.set_boundary(mpath.Path(verts * 0.5 + [0.5, 0.5]), transform=ax.transAxes)
    gl = ax.gridlines(linewidth=0.5, color="gray", alpha=0.7, linestyle="--")
    gl.xlocator = mticker.FixedLocator(np.arange(-180, 181, 60))
    gl.ylocator = mticker.FixedLocator([30, 50, 70])
    lat_formatter = LatitudeFormatter()
    lon_formatter = LongitudeFormatter()
    for lat in [30, 50, 70]:
        ax.text(
            180, lat + 5, lat_formatter(lat),
            transform=ccrs.PlateCarree(), ha="left", va="center",
            fontsize=12, color="black",
        )
    for lon in np.arange(-180, 181, 60):
        if lon == -180:  # same meridian as 180°
            continue
        label_lon, label_lat = lon, 25
        if lon in [-120, 120]:
            label_lat = 23
        if lon == -60:
            label_lat = 25
        if lon == 0:
            label_lon = 2
            label_lat = 29
        if lon == 180:
            label_lon = 178
        ax.text(
            label_lon, label_lat, lon_formatter(lon),
            transform=ccrs.PlateCarree(),
            ha="center", va="top", fontsize=12, color="black",
        )

def show_metric_maps_by_model(
    df_long,
    metric="rmse",
    models=("CDD", "CDDP", "SIAM", "SIAMP"),
    cmap=None,
    vmin=None,
    vmax=None,
    cbar_ticks=None,
    cbar_label=None,
    figsize=(6, 6),
):
    setup = _polar_land_setup()
    models = [m for m in models if m in set(df_long["model"])]
    vals = df_long.loc[df_long["model"].isin(models), metric].to_numpy(float)
    vals = vals[np.isfinite(vals)]
    if vmin is None:
        vmin = float(np.nanpercentile(vals, 2))
    if vmax is None:
        vmax = float(np.nanpercentile(vals, 98))
    if cmap is None:
        cmap = FIG1_CMAP
    if cbar_label is None:
        cbar_label = {"rmse": "RMSE (days)", "kge": "KGE", "r": "r"}[metric]

    norm = Normalize(vmin=vmin, vmax=vmax)
    figs = {}
    for mname in models:
        sub = df_long[df_long["model"] == mname][["latitude", "longitude", metric]].copy()
        rast = rasterize_metric(sub, metric, setup)

        fig = plt.figure(figsize=figsize)
        ax = fig.add_subplot(1, 1, 1, projection=ccrs.NorthPolarStereo())
        _decorate_polar(ax)
        im = ax.imshow(
            np.ma.masked_invalid(rast),
            cmap=cmap,
            norm=norm,
            extent=[setup["left"], setup["right"], setup["bottom"], setup["top"]],
            transform=ccrs.PlateCarree(),
            origin="upper",
            interpolation="nearest",
        )
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.08, shrink=0.75)
        cbar.set_label(cbar_label, fontsize=11)
        if cbar_ticks is not None:
            cbar.set_ticks(list(cbar_ticks))
        figs[mname] = fig
    return figs

OUT_DIR = Path("../../results/si_figures/si_fig5")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("OUT_DIR:", OUT_DIR.resolve())

def show_p_vs_base_diff_maps(
    df_long,
    metric="rmse",
    pairs=(("CDD", "CDDP"), ("SIAM", "SIAMP")),
    vmin=None,
    vmax=None,
    cbar_ticks=None,
    figsize=(6, 6),
):
    from matplotlib.colors import TwoSlopeNorm

    setup = _polar_land_setup()
    wide = (
        df_long.pivot_table(
            index=["pixel", "latitude", "longitude"],
            columns="model",
            values=metric,
            aggfunc="first",
        )
        .reset_index()
    )

    pair_cols = []
    for base, pmodel in pairs:
        if base not in wide.columns or pmodel not in wide.columns:
            print(f"Skip missing pair {base}/{pmodel}")
            continue
        col = f"{base}_to_{pmodel}"
        if metric == "rmse":
            wide[col] = wide[base] - wide[pmodel]
            cbar_label = "ΔRMSE (days)\n(base − P; >0 = P better)"
        elif metric == "kge":
            wide[col] = wide[pmodel] - wide[base]
            cbar_label = "ΔKGE\n(P − base; >0 = P better)"
        else:
            raise ValueError(metric)
        pair_cols.append((col, base, pmodel, cbar_label))

    vals = np.concatenate([wide[c].to_numpy(float) for c, _, _, _ in pair_cols])
    vals = vals[np.isfinite(vals)]
    if vmin is None or vmax is None:
        lim = float(np.nanpercentile(np.abs(vals), 98))
        lim = max(lim, 1e-6)
        vmin = -lim if vmin is None else vmin
        vmax = lim if vmax is None else vmax
    lim = max(abs(float(vmin)), abs(float(vmax)))
    vmin, vmax = -lim, lim
    if cbar_ticks is None:
        step = lim / 2.0
        cbar_ticks = [-lim, -step, 0.0, step, lim]

    cmap = plt.cm.RdBu_r
    norm = TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)

    figs = {}
    for col, base, pmodel, cbar_label in pair_cols:
        sub = wide[["latitude", "longitude", col]].copy()
        rast = rasterize_metric(sub, col, setup)
        fig = plt.figure(figsize=figsize)
        ax = fig.add_subplot(1, 1, 1, projection=ccrs.NorthPolarStereo())
        _decorate_polar(ax)
        im = ax.imshow(
            np.ma.masked_invalid(rast),
            cmap=cmap,
            norm=norm,
            extent=[setup["left"], setup["right"], setup["bottom"], setup["top"]],
            transform=ccrs.PlateCarree(),
            origin="upper",
            interpolation="nearest",
        )
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.08, shrink=0.75)
        cbar.set_label(cbar_label, fontsize=11)
        cbar.set_ticks(list(cbar_ticks))
        figs[f"{base}_to_{pmodel}"] = fig
    return figs


In [ ]:
figs_rmse = show_metric_maps_by_model(
    evaluation_results, metric="rmse",
    vmin=4, vmax=17, cbar_ticks=[5, 10, 15],
)
for mname, fig in figs_rmse.items():
    fp = OUT_DIR / f"rmse_{mname}.png"
    fig.savefig(fp, dpi=300, bbox_inches="tight")
    print("Saved", fp)
    plt.show()


In [ ]:
figs_kge = show_metric_maps_by_model(
    evaluation_results, metric="kge",
    vmin=0, vmax=0.65, cbar_ticks=[0, 0.2, 0.4, 0.6],
)
for mname, fig in figs_kge.items():
    fp = OUT_DIR / f"kge_{mname}.png"
    fig.savefig(fp, dpi=300, bbox_inches="tight")
    print("Saved", fp)
    plt.show()


## P vs base skill


In [ ]:
figs_drmse = show_p_vs_base_diff_maps(
    evaluation_results, metric="rmse",
    vmin=-1.5, vmax=1.5, cbar_ticks=[-1.5, -0.75, 0, 0.75, 1.5],
)
for key, fig in figs_drmse.items():
    fp = OUT_DIR / f"rmse_diff_{key}.png"
    fig.savefig(fp, dpi=300, bbox_inches="tight")
    print("Saved", fp)
    plt.show()

figs_dkge = show_p_vs_base_diff_maps(
    evaluation_results, metric="kge",
    vmin=-0.25, vmax=0.25, cbar_ticks=[-0.25, -0.125, 0, 0.125, 0.25],
)
for key, fig in figs_dkge.items():
    fp = OUT_DIR / f"kge_diff_{key}.png"
    fig.savefig(fp, dpi=300, bbox_inches="tight")
    print("Saved", fp)
    plt.show()
